In [4]:
# If needed (in a notebook): !pip install pulp

import pulp

# Data
teachers = {"Alpha": 200, "Beta": 120}  # x1
spending = {"Alpha": 8,   "Beta": 12}   # x2
graduation = {"Alpha": 85, "Beta": 86}  # y

def dea_with_pulp(unit_name, teachers, spending, graduation, solver=None):
    """
    Solve the DEA multiplier model for the specified unit (unit_name) using PuLP.
    Variables: u (output weight), v1 (weight for teachers), v2 (weight for spending).
    Objective: max u * y_unit
    Subject to:
        For all j: u*y_j <= v1*x1_j + v2*x2_j
        Normalization: v1*x1_unit + v2*x2_unit = 1
        u, v1, v2 >= 0
    Returns: (efficiency, u, v1, v2)
    """
    # LP: maximize
    model = pulp.LpProblem(f"DEA_{unit_name}", pulp.LpMaximize)

    # Variables (nonnegative)
    u  = pulp.LpVariable("u",  lowBound=0)
    v1 = pulp.LpVariable("v1", lowBound=0)
    v2 = pulp.LpVariable("v2", lowBound=0)

    # Objective: maximize u * y_unit
    model += u * graduation[unit_name]

    # Constraints: for all j, u*y_j <= v1*x1_j + v2*x2_j
    for j in teachers.keys():
        model += u * graduation[j] <= v1 * teachers[j] + v2 * spending[j], f"Eff_{j}_le_1"

    # Normalization: v1*x1_unit + v2*x2_unit = 1
    model += v1 * teachers[unit_name] + v2 * spending[unit_name] == 1, "Normalization"

    # Solve
    if solver is None:
        solver = pulp.PULP_CBC_CMD(msg=False)
    model.solve(solver)

    if pulp.LpStatus[model.status] != "Optimal":
        raise RuntimeError(f"Optimization for {unit_name} not optimal: {pulp.LpStatus[model.status]}")

    eff = pulp.value(model.objective)
    return eff, u.value(), v1.value(), v2.value()

# Run for Alpha and Beta
eff_alpha, u_a, v1_a, v2_a = dea_with_pulp("Alpha", teachers, spending, graduation)
eff_beta,  u_b, v1_b, v2_b  = dea_with_pulp("Beta",  teachers, spending, graduation)

print("=== Alpha ===")
print(f"Efficiency (h0): {eff_alpha:.6f}")
print(f"u={u_a:.6f}, v1={v1_a:.6f}, v2={v2_a:.6f}")

print("\n=== Beta ===")
print(f"Efficiency (h0): {eff_beta:.6f}")
print(f"u={u_b:.6f}, v1={v1_b:.6f}, v2={v2_b:.6f}")


=== Alpha ===
Efficiency (h0): 1.000000
u=0.011765, v1=0.000000, v2=0.125000

=== Beta ===
Efficiency (h0): 1.000000
u=0.011628, v1=0.008333, v2=0.000000


In [5]:
# !pip install pulp

import pulp

# Data
x1 = {"A": 10, "B": 0,  "C": 5}
x2 = {"A": 0,  "B": 10, "C": 5}
y  = {"A": 10, "B": 10, "C": 8}

unit = "C"  # evaluate C

# LP model: maximize u*y_C
model = pulp.LpProblem("DEA_C", pulp.LpMaximize)
u  = pulp.LpVariable("u",  lowBound=0)
v1 = pulp.LpVariable("v1", lowBound=0)
v2 = pulp.LpVariable("v2", lowBound=0)

# Objective
model += u * y[unit]

# Constraints: for all j, u*y_j <= v1*x1_j + v2*x2_j
for j in ["A","B","C"]:
    model += u * y[j] <= v1 * x1[j] + v2 * x2[j], f"Eff_{j}_le_1"

# Normalization on C
model += v1 * x1[unit] + v2 * x2[unit] == 1

# Solve
model.solve(pulp.PULP_CBC_CMD(msg=False))
print("Status:", pulp.LpStatus[model.status])
print("Max efficiency h_C =", pulp.value(model.objective))
print("u =", u.value(), "v1 =", v1.value(), "v2 =", v2.value())


Status: Optimal
Max efficiency h_C = 0.8
u = 0.1 v1 = 0.1 v2 = 0.1


In [6]:
# If needed (in a notebook/CLI):  !pip install pulp
import numpy as np
import pulp

# --------------------------
# 1) Make example data
# --------------------------
rng = np.random.default_rng(42)

n_units = 6      # number of countries/DMUs
m_inputs = 10    # number of inputs
s_outputs = 5    # number of outputs

# Positive inputs/outputs
X = rng.uniform(10, 100, size=(n_units, m_inputs))   # inputs  (n x m)
Y = rng.uniform(20, 200, size=(n_units, s_outputs))  # outputs (n x s)

DMUs = [f"U{i+1}" for i in range(n_units)]

# --------------------------
# 2) DEA (CCR, output-oriented) with PuLP
# --------------------------
def dea_ccr_output_pulp(dmu_idx, X, Y):
    """
    Solve multiplier-form DEA (CCR, output-oriented) for DMU 'dmu_idx'.
    Maximize sum_r u_r * y_{r,o}
    s.t. for all j: sum_r u_r * y_{r,j} <= sum_i v_i * x_{i,j}
         normalization: sum_i v_i * x_{i,o} = 1
         u_r, v_i >= 0
    Returns: efficiency, u (outputs weights), v (inputs weights)
    """
    n, m = X.shape[0], X.shape[1]
    s = Y.shape[1]
    o = dmu_idx

    model = pulp.LpProblem(f"DEA_CCR_Output_DMUid_{o}", pulp.LpMaximize)

    # Variables
    u = [pulp.LpVariable(f"u_{r}", lowBound=0) for r in range(s)]
    v = [pulp.LpVariable(f"v_{i}", lowBound=0) for i in range(m)]

    # Objective: maximize weighted outputs of DMU o
    model += pulp.lpSum(u[r] * Y[o, r] for r in range(s))

    # Constraints: for all j, weighted outputs <= weighted inputs
    for j in range(n):
        model += pulp.lpSum(u[r] * Y[j, r] for r in range(s)) <= \
                 pulp.lpSum(v[i] * X[j, i] for i in range(m)), f"Eff_le_1_j{j}"

    # Normalization on DMU o inputs
    model += pulp.lpSum(v[i] * X[o, i] for i in range(m)) == 1, "Normalization"

    # Solve
    model.solve(pulp.PULP_CBC_CMD(msg=False))

    if pulp.LpStatus[model.status] != "Optimal":
        raise RuntimeError(f"Not optimal for DMU {o}: {pulp.LpStatus[model.status]}")

    eff = pulp.value(model.objective)
    u_vals = np.array([var.value() for var in u])
    v_vals = np.array([var.value() for var in v])
    return eff, u_vals, v_vals

# --------------------------
# 3) Run DEA for all DMUs
# --------------------------
results = []
for idx, name in enumerate(DMUs):
    eff, u_w, v_w = dea_ccr_output_pulp(idx, X, Y)
    results.append((name, eff, u_w, v_w))

# --------------------------
# 4) Display results
# --------------------------
for name, eff, u_w, v_w in results:
    print(f"{name}: efficiency = {eff:.6f}")
    print(f"  output weights u (len={len(u_w)}): {np.round(u_w, 6)}")
    print(f"  input  weights v (len={len(v_w)}): {np.round(v_w, 6)}\n")


U1: efficiency = 1.000000
  output weights u (len=5): [0.       0.       0.       0.006341 0.      ]
  input  weights v (len=10): [0.       0.005829 0.       0.       0.       0.007274 0.       0.
 0.       0.      ]

U2: efficiency = 1.000000
  output weights u (len=5): [0.007836 0.       0.       0.       0.000633]
  input  weights v (len=10): [0.001409 0.       0.       0.       0.       0.       0.       0.010856
 0.00909  0.      ]

U3: efficiency = 1.000000
  output weights u (len=5): [0.      0.      0.00576 0.      0.     ]
  input  weights v (len=10): [0.       0.       0.       0.       0.       0.008724 0.       0.
 0.       0.010632]

U4: efficiency = 1.000000
  output weights u (len=5): [0.       0.       0.       0.       0.006207]
  input  weights v (len=10): [0.       0.008231 0.       0.       0.       0.007428 0.       0.
 0.       0.      ]

U5: efficiency = 1.000000
  output weights u (len=5): [0.007165 0.       0.       0.       0.      ]
  input  weights v (len=10